In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a href="https://colab.research.google.com/github/lmoroney/dlaicourse/blob/master/TensorFlow%20In%20Practice/Course%203%20-%20NLP/Course%203%20-%20Week%204%20-%20Lesson%202%20-%20Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Imports: pad_sequences (iguala o tamanho das sequências), Embedding/LSTM/Dense/Bidirectional
# (camadas), Tokenizer (texto -> números), Sequential, Adam e numpy.
import tensorflow as tf

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
import numpy as np

In [ ]:
# Baixa o dataset de letras de músicas irlandesas pra /tmp ('!' = comando de shell). Precisa de
# internet.
!wget --no-check-certificate \
    https://storage.googleapis.com/learning-datasets/irish-lyrics-eof.txt \
    -O /tmp/irish-lyrics-eof.txt

In [ ]:
# Tokeniza o corpus: cada palavra vira um inteiro.
tokenizer = Tokenizer()

# Lê o arquivo inteiro como uma string.
data = open('/tmp/irish-lyrics-eof.txt').read()

# corpus = lista de linhas em minúsculo (cada linha é um verso).
corpus = data.lower().split("\n")

# Constrói o vocabulário (palavra -> id).
tokenizer.fit_on_texts(corpus)
# +1 porque o índice 0 é reservado pro padding.
total_words = len(tokenizer.word_index) + 1

print(tokenizer.word_index)
print(total_words)


In [ ]:
# Monta os dados de treino em n-gramas: pra cada verso, gera prefixos cada vez maiores. Ex: 'in
# the town' -> [in the], [in the town]. A última palavra de cada sequência é o alvo.
input_sequences = []
for line in corpus:
	# Verso -> lista de ids.
	token_list = tokenizer.texts_to_sequences([line])[0]
	for i in range(1, len(token_list)):
		# Prefixo do verso até a posição i (inclui a palavra que vai ser prevista).
		n_gram_sequence = token_list[:i+1]
		input_sequences.append(n_gram_sequence)

# pad sequences
# Tamanho da maior sequência.
max_sequence_len = max([len(x) for x in input_sequences])
# padding='pre': completa com zeros à ESQUERDA, pra todas ficarem do mesmo tamanho.
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# create predictors and label
# xs = todas as palavras menos a última (entrada); labels = a última palavra (o que o modelo deve
# prever).
xs, labels = input_sequences[:,:-1],input_sequences[:,-1]

# One-hot: cada label vira um vetor de tamanho total_words (classificação multiclasse com
# softmax).
ys = tf.keras.utils.to_categorical(labels, num_classes=total_words)

In [ ]:
# Confere o id de algumas palavras do vocabulário (palavras do primeiro verso do dataset).
print(tokenizer.word_index['in'])
print(tokenizer.word_index['the'])
print(tokenizer.word_index['town'])
print(tokenizer.word_index['of'])
print(tokenizer.word_index['athy'])
print(tokenizer.word_index['one'])
print(tokenizer.word_index['jeremy'])
print(tokenizer.word_index['lanigan'])

In [ ]:
# Exemplo de entrada de treino (índice 6): ids com zeros de padding à esquerda.
print(xs[6])

In [ ]:
# Alvo one-hot desse exemplo: tudo zero, exceto 1 na posição da próxima palavra.
print(ys[6])

In [ ]:
# Compare com o exemplo 6: cada linha seguinte acrescenta a próxima palavra do verso.
print(xs[5])
print(ys[5])

In [ ]:
# Dicionário completo palavra -> id.
print(tokenizer.word_index)

In [ ]:
# Define, compila e treina o modelo de geração de texto por palavras.
# Modelo sequencial: camadas empilhadas em ordem.
model = Sequential()
# Embedding: cada palavra vira um vetor denso de 100 dimensões. input_length = tamanho das
# entradas (max_len - 1). Obs: em versões novas do Keras o input_length pode gerar aviso.
model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))
# LSTM bidirecional com 150 unidades: lê a sequência nos dois sentidos.
model.add(Bidirectional(LSTM(150)))
# Saída: distribuição de probabilidade (softmax) sobre todo o vocabulário.
model.add(Dense(total_words, activation='softmax'))
adam = Adam(lr=0.01)
# categorical_crossentropy porque os alvos (ys) estão em one-hot.
model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])
#earlystop = EarlyStopping(monitor='val_loss', min_delta=0, patience=5, verbose=0, mode='auto')
# Treina por 100 épocas; o history guarda as métricas de cada época.
history = model.fit(xs, ys, epochs=100, verbose=1)
#print model.summary()
# Imprime só o objeto do modelo. Pra ver camadas e parâmetros, use model.summary().
print(model)


In [ ]:
# Função utilitária que plota uma métrica do history (ex: 'accuracy') ao longo das épocas.
import matplotlib.pyplot as plt


def plot_graphs(history, string):
  plt.plot(history.history[string])
  plt.xlabel("Epochs")
  plt.ylabel(string)
  plt.show()

In [ ]:
# Gráfico da acurácia de treino.
plot_graphs(history, 'accuracy')


In [ ]:
# Gera texto: a partir de uma seed, prevê a próxima palavra 100 vezes, sempre realimentando o
# modelo com o texto gerado até ali.
# Texto inicial. Palavras fora do vocabulário são ignoradas pelo tokenizer.
seed_text = "I've got a bad feeling about this"
# Quantas palavras gerar.
next_words = 100

for _ in range(next_words):
	# Texto atual -> ids.
	token_list = tokenizer.texts_to_sequences([seed_text])[0]
	# Completa com zeros à esquerda pro tamanho que o modelo espera.
	token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
	# Pega o id da palavra mais provável (argmax é determinístico, por isso o texto tende a ficar
	# repetitivo).
	predicted = np.argmax(model.predict(token_list), axis=-1)
	output_word = ""
	# Procura no word_index a palavra com esse id (busca linear; dava pra inverter o dicionário).
	for word, index in tokenizer.word_index.items():
		if index == predicted:
			output_word = word
			break
	# Acrescenta a palavra gerada e repete usando o texto inteiro como novo contexto.
	seed_text += " " + output_word
print(seed_text)